In [1]:
from collections import Counter
from itertools import chain
import nltk
from nltk.lm import Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import Vocabulary
import pandas as pd
from pathlib import Path
import re

In [2]:
# Find all CSV files in the data folder

DATA_PATH = Path("../data/Translated")
csv_files = sorted(DATA_PATH.glob("*.csv"))

print(f"Found {len(csv_files)} files:")
for f in csv_files:
    print(f.name)

Found 15 files:
Speeches_CZ_Cleaned_Translated.csv
Speeches_DE_Cleaned_Translated.csv
Speeches_ECB_Cleaned_Translated.csv
Speeches_ECR_Cleaned_Translated.csv
Speeches_EC_Cleaned_Translated.csv
Speeches_EP_Cleaned_Translated.csv
Speeches_EUCouncil_Cleaned_Translated.csv
Speeches_GR_Cleaned_Translated.csv
Speeches_IMF_Cleaned_Translated.csv
Speeches_IT_Cleaned_Translated.csv
Speeches_NL_Cleaned_Translated.csv
Speeches_PL_Cleaned_Translated.csv
Speeches_PO_Cleaned_Translated.csv
Speeches_SP_Cleaned_Translated.csv
Speeches_UK_Cleaned_Translated.csv


In [3]:
# Load all CSV files into DataFrame

ALL_COLUMNS = ["title", "date", "location", "speaker", "word_count", "raw", "raw_text", "original", "lang"]
USE_COLUMNS = ["title", "date", "location", "speaker", "raw"]

def clean_sentence(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_sentences(text: str) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [clean_sentence(s) for s in sentences if s.strip()]

def load_speech_csv(path: Path) -> pd.DataFrame:
    nation = path.stem.split("_")[1]
    df = pd.read_csv(path, header=None, names=ALL_COLUMNS, usecols=USE_COLUMNS)
    df["nation"] = nation
    df["sentences"] = df["raw"].apply(split_sentences)
    return df

corpus = pd.concat([load_speech_csv(f) for f in csv_files], ignore_index=True)
print(f"Loaded {len(corpus)} speeches:")
corpus.head()

Loaded 5029 speeches:


,title,date,location,speaker,raw,nation,sentences
0,Rede von Guy Verhofstadt anläßlich des FDP Eur...,23-06-2013,European Parliament,Guy Verhofstadt,"Love Nadja, dear friends,Dear Sir or Madam,rea...",CZ,[love nadja dear friends dear sir or madam rea...
1,Speech by Guy Verhofstadt - Intervention in pl...,05-02-2013,European Parliament,Guy Verhofstadt,Intervention Guy VerhofstadtNovember 21 2012Vi...,CZ,[intervention guy verhofstadtnovember 21 2012v...
2,Speech by Guy Verhofstadt on the future of Europe,09-05-2012,European Parliament,Guy Verhofstadt,"Dear President, Dear colleagues,Let me paraphr...",CZ,[dear president dear colleagues let me paraphr...
3,Speech by Guy Verhofstadt on PPV,12-03-2012,European Parliament,Guy Verhofstadt,"My group, President, for the initiative of Mr ...",CZ,[my group president for the initiative of mr w...
4,Le Grand Direct de Guy Verhofstadt,07-03-2012,European Parliament,Guy Verhofstadt,Guy Verhofstadt Live dupuis RTL-TVI answering ...,CZ,[guy verhofstadt live dupuis rtl tvi answering...


In [4]:
corpus.groupby("nation").size().sort_values(ascending=False).rename("speech_count")

nation
EUCouncil    1064
UK            996
NL            804
ECB           580
GR            390
IMF           390
PL            260
DE            234
EP            149
SP            133
ECR            18
CZ              6
PO              4
EC              1
Name: speech_count, dtype: int64

In [5]:

def tokenize_sentences(sentences: list[str]) -> list[list[str]]:
    return [s.split() for s in sentences if s.strip()]

corpus["tokens"] = corpus["sentences"].apply(tokenize_sentences)

In [6]:
all_tokens = chain.from_iterable(
    chain.from_iterable(token_lists) for token_lists in corpus["tokens"]
)
universal_vocab = Vocabulary(all_tokens, unk_cutoff=1)

print(f"Universal vocabulary size: {len(universal_vocab):,} tokens")

Universal vocabulary size: 51,441 tokens


In [7]:
N = 2
nation_models: dict[str, Laplace] = {}

for nation, group in corpus.groupby("nation"):
    nation_sentences = list(chain.from_iterable(group["tokens"]))
    train_data, _ = padded_everygram_pipeline(N, nation_sentences)

    model = Laplace(N, vocabulary=universal_vocab)
    model.fit(train_data)

    nation_models[nation] = model
    print(f"{nation:>10}: {len(nation_sentences):>7,} sentences")

print("-" * 30)
print(f"Built {len(nation_models)} nation-level {N}-gram models")

        CZ:     246 sentences
        DE:   4,935 sentences
        EC:      95 sentences
       ECB:  59,855 sentences
       ECR:   1,469 sentences
        EP:   9,541 sentences
 EUCouncil:  12,620 sentences
        GR:  43,631 sentences
       IMF:  43,631 sentences
        NL:  42,070 sentences
        PL:  15,475 sentences
        PO:   1,017 sentences
        SP:   9,053 sentences
        UK: 119,197 sentences
------------------------------
Built 14 nation-level 2-gram models


In [8]:
test_context = ["european"]
test_word = "union"

print(f"P('{test_word}' | '{test_context[-1]}')")
print("-" * 30)
for nation, model in sorted(nation_models.items()):
    score = model.score(test_word, test_context)
    print(f"{nation:>10}: {score:.6f}")

P('union' | 'european')
------------------------------
        CZ: 0.000175
        DE: 0.006866
        EC: 0.000019
       ECB: 0.021335
       ECR: 0.000310
        EP: 0.007219
 EUCouncil: 0.012881
        GR: 0.012621
       IMF: 0.012621
        NL: 0.013750
        PL: 0.001869
        PO: 0.000408
        SP: 0.001633
        UK: 0.056442


In [11]:
#exporting corpus for use with SRILM

output_directory = Path("../output/countries")
output_directory.mkdir(parents=True, exist_ok=True)

for nation, group in corpus.groupby("nation"):
    out_file = output_directory / f"{nation}.txt"
    
    with open(out_file, "w", encoding="utf-8") as f:
        for sentence_list in group["sentences"]:
            for sent in sentence_list:
                f.write(sent + "\n")